In [ ]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [3]:
#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://localhost:4566'
# os.chdir('/config/workspace/python/pythonawsproject')
# print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

{}


In [4]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [5]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

Created VPC: vpc-26e93038
VPC is now avilable
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State saved
Result: vpc-26e93038


In [6]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
{'Subnet': {'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '000000000000', 'AssignIpv6AddressOnCreation': False, 'Ipv6CidrBlockAssociationSet': [], 'Tags': [], 'SubnetArn': 'arn:aws:ec2:ap-south-1:000000000000:subnet/subnet-57c10297', 'EnableDns64': False, 'SubnetId': 'subnet-57c10297', 'State': 'available', 'VpcId': 'vpc-26e93038', 'CidrBlock': '10.0.1.0/24', 'AvailableIpAddressCount': 251, 'AvailabilityZone': 'ap-south-1a', 'DefaultForAz': False, 'MapPublicIpOnLaunch': False}, 'ResponseMetadata': {'RequestId': '85c30881-b539-437e-a12f-1488a452733d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'content-length': '862', 'content-type': 'application/xml', 'date

In [7]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")

Created IGW: igw-55c22648
Attachment IGW to VPC: vpc-26e93038
Tagged  IGW: dev-igw
Saved IGW to State
Result: igw-55c22648


In [8]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Created public route table: rtb-bf913ca7
Added route: '0.0.0.0/0' rtb-bf913ca7
Public route table already exists: rtb-bf913ca7
Created private route table: rtb-8b9e29c2
Saved private route table to state
  Associated private-subnet-az1 → private route table
  Associated private-subnet-az2 → private route table
Route tables done.


In [9]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Allocated Elastic IP: eipalloc-266899f506056744e
Created NAT Gateway: nat-d4b88b28da0f5ef08
Waiting for NAT Gateway to become available (1-2 min)...
NAT Gateway is now available
Saved NAT Gateway to state
Added route: 0.0.0.0/0 → nat-d4b88b28da0f5ef08 in private route table
NAT Gateway done.


In [10]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [12]:
import os
os.chdir('/workspaces/floci-boto-3-python')
print(os.listdir('network/'))

['ec2_instance.py', 'state.py', 'main.py', '__pycache__', 'nat_gateway.py', 'security_groups.py', 'internet_gateway.py', 'subnets.py', '__init__.py', 'route_tables.py', 'aws_client.py', 'planner.py', 'vpc.py']


In [13]:
from network.security_groups import create_security_groups
create_security_groups(config)

Created Security Group: sg-8373b0b2bb41811c5 (dev-web-sg)
  Added 3 ingress rules
  Revoked default egress (allow-all) rule
  Tagged + saved: sg_dev-web-sg = sg-8373b0b2bb41811c5
Security Groups done.


{'vpc_id': 'vpc-26e93038',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-57c10297',
 'public-subnet-az2': 'subnet-0d582675',
 'private-subnet-az1': 'subnet-9890b6ca',
 'private-subnet-az2': 'subnet-97d5730e',
 'igw_id': 'igw-55c22648',
 'public_rt_id': 'rtb-bf913ca7',
 'public-subnet-az1_rt_assoc': 'rtbassoc-58fabcad',
 'public-subnet-az2_rt_assoc': 'rtbassoc-4f2c9b7d',
 'private_rt_id': 'rtb-8b9e29c2',
 'private-subnet-az1_rt_assoc': 'rtbassoc-ea3977d7',
 'private-subnet-az2_rt_assoc': 'rtbassoc-6ed2ba01',
 'nat_eip_alloc_id': 'eipalloc-266899f506056744e',
 'nat_id': 'nat-d4b88b28da0f5ef08',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-8373b0b2bb41811c5'}

In [14]:

from network.ec2_instance import create_ec2_instance
instance_id = create_ec2_instance(config)
print(f"Result: {instance_id}")

Launched EC2 instance: i-f7e7b7ef478456c92
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-f7e7b7ef478456c92
Result: i-f7e7b7ef478456c92


In [15]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-26e93038...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-57c10297...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-0d582675...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-9890b6ca...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-97d5730e...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-55c22648...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-d4b88b28da0f5ef0...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-bf913ca7...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-8b9e29c2...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-8373b0b2bb41811c5...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [16]:
from network.main import command_plan
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-26e93038...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-57c10297...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-0d582675...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-9890b6ca...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-97d5730e...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-55c22648...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-d4b88b28da0f5ef0...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-bf913ca7...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-8b9e29c2...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-8373b0b2bb41811c5...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-26e93038'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-57c10297'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-0d582675'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-9890b6ca'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-97d5730e'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-55c22648'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-d4b88b28da0f5ef08'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-bf913ca7'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-8b9e29c2'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-8373b0b2bb41811c5'}]

In [ ]:
from network.main import command_destroy
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

----Terminate EC2 Instance------
  Waiting for i-f7e7b7ef478456c92 to terminate...
  EC2 instance terminated.
===== Destroying NAT Gateway =====
  Waiting for NAT nat-d4b88b28da0f5ef08 to die...


In [ ]:
command_plan(config)